In [ ]:
from torch import nn
import torch
from einops.layers.torch import Rearrange, Reduce


class PatchEmbedding(nn.Module):
    def __init__(self, emb_size=40, n_channels=63):
        super().__init__()
        # Revised from ShallowNet
        self.tsconv = nn.Sequential(
            nn.Conv2d(1, 40, (1, 25), stride=(1, 1)),
            nn.AvgPool2d((1, 51), (1, 5)),
            nn.BatchNorm2d(40),
            nn.ELU(),
            nn.Conv2d(40, 40, (n_channels, 1), stride=(1, 1)),
            nn.BatchNorm2d(40),
            nn.ELU(),
            nn.Dropout(0.5),
        )

        self.projection = nn.Sequential(
            nn.Conv2d(40, emb_size, (1, 1), stride=(1, 1)),
            Rearrange("b e (h) (w) -> b (h w) e"),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # b, _, _, _ = x.shape
        x = x.unsqueeze(1)
        # print("x", x.shape)
        x = self.tsconv(x)
        # print("tsconv", x.shape)
        x = self.projection(x)
        # print("projection", x.shape)
        return x
    
E = PatchEmbedding(H=32)
E.to("cuda")

PatchEmbedding(
  (tsconv): Sequential(
    (0): Conv2d(1, 40, kernel_size=(1, 25), stride=(1, 1))
    (1): AvgPool2d(kernel_size=(1, 51), stride=(1, 5), padding=0)
    (2): BatchNorm2d(40, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): ELU(alpha=1.0)
    (4): Conv2d(40, 40, kernel_size=(32, 1), stride=(1, 1))
    (5): BatchNorm2d(40, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ELU(alpha=1.0)
    (7): Dropout(p=0.5, inplace=False)
  )
  (projection): Sequential(
    (0): Conv2d(40, 40, kernel_size=(1, 1), stride=(1, 1))
    (1): Rearrange('b e (h) (w) -> b (h w) e')
  )
)

In [10]:
x = torch.randn((1, 32, 250)).to("cuda")

print(E(x).shape)


torch.Size([1, 36, 40])


In [11]:
36*40

1440